# Module 2 - Vector Search: Homework

We use a lightweight ONNX embedder (no PyTorch needed) to turn text into vectors, then explore different search strategies.

## Setup

Run this **once** in your terminal (not here) to install dependencies:

```bash
pip install onnxruntime tokenizers numpy tqdm minsearch gitsource
```

Then download the two helper scripts that wrap the ONNX model:

In [1]:
import urllib.request, os

PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"

for fname in ["download.py", "embedder.py"]:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(f"{PREFIX}/{fname}", fname)
        print(f"Downloaded {fname}")
    else:
        print(f"{fname} already exists")

download.py already exists
embedder.py already exists


In [2]:
# Downloads the ONNX model weights (~22 MB). Run once.
%run download.py

C:\Users\hiago.garcia\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  exists models\Xenova\all-MiniLM-L6-v2\tokenizer.json
  exists models\Xenova\all-MiniLM-L6-v2\model.onnx


## Load the Embedder

The `Embedder` class wraps the `all-MiniLM-L6-v2` model in ONNX format.  
It turns any string into a **384-dimensional vector** — a list of 384 numbers that captures the meaning of the text.

In [3]:
from embedder import Embedder

embedder = Embedder()
print("Embedder ready")

Embedder ready


---
## Q1 — Embedding a Query

**Key concept:** An embedding converts text into a fixed-size vector. Similar texts get similar vectors — that's how semantic search works.

We embed one query and look at its first number.

In [4]:
query = "How does approximate nearest neighbor search work?"
query_vector = embedder.encode(query)

print(f"Vector length: {len(query_vector)}")
print(f"First value:   {query_vector[0]:.4f}")

Vector length: 384
First value:   -0.0206


**Answer Q1:** The first value of the query vector.

---
## Q2 — Cosine Similarity

**Key concept:** Cosine similarity measures the *angle* between two vectors — 1.0 means identical direction (very similar meaning), 0 means unrelated.  
When vectors are **normalized** (length = 1), cosine similarity is just the **dot product** — multiply element-by-element and sum.

We embed a specific lesson page and measure how similar it is to our query.

In [5]:
from gitsource import GithubRepositoryDataReader
import numpy as np

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: path == "02-vector-search/lessons/07-sqlitesearch-vector.md",
)

files = list(reader.read())
doc = files[0].parse()
print(f"Loaded: {doc['filename']}")
print(f"Content length: {len(doc['content'])} chars")

Loaded: 02-vector-search/lessons/07-sqlitesearch-vector.md
Content length: 7219 chars


In [6]:
doc_vector = embedder.encode(doc["content"])

# Dot product = cosine similarity when vectors are already normalized by the embedder
similarity = np.dot(query_vector, doc_vector)
print(f"Cosine similarity: {similarity:.4f}")

Cosine similarity: 0.3611


**Answer Q2:** The cosine similarity value above.

---
## Load All Documents

Now we grab all 72 lesson pages from a pinned commit of the course repo.  
Using a fixed `commit_id` ensures everyone gets the same data.

In [7]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} documents")

Loaded 72 documents


---
## Q3 — Chunking and Manual Search

**Key concept:** Long documents don't fit well in a single embedding — the model has a token limit and details get lost.  
**Chunking** splits documents into overlapping windows (`size=2000` chars, `step=1000` means 50% overlap).  
The overlap ensures a sentence near a boundary still appears fully in at least one chunk.

We embed every chunk, then score each against our query using the dot product.

In [8]:
from gitsource import chunk_documents
from tqdm import tqdm

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")

Total chunks: 295


In [9]:
# Embed every chunk — this is the expensive step (~minutes depending on hardware)
for chunk in tqdm(chunks):
    chunk["vector"] = embedder.encode(chunk["content"])

100%|██████████| 295/295 [00:03<00:00, 83.67it/s]


In [10]:
# Score each chunk against the query using dot product
for chunk in chunks:
    chunk["score"] = np.dot(query_vector, chunk["vector"])

best = max(chunks, key=lambda c: c["score"])
print(f"Best chunk score:  {best['score']:.4f}")
print(f"From file:         {best['filename']}")

Best chunk score:  0.6489
From file:         02-vector-search/lessons/07-sqlitesearch-vector.md


**Answer Q3:** The filename of the best-scoring chunk.

---
## Q4 — Vector Search with minsearch

**Key concept:** Instead of looping manually, a vector search library builds an **index** — a data structure optimized to find nearest neighbors fast (no need to score every chunk one by one).

`minsearch.VectorSearch` is a simple in-memory version of this.

In [ ]:
import minsearch
import numpy as np

vectors = np.array([chunk["vector"] for chunk in chunks])

vs = minsearch.VectorSearch()
vs.fit(vectors, chunks)  # vectors: 2D array (n_chunks, 384); payload: the chunk dicts
print("Index built")

In [ ]:
query4 = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(query4)

results4 = vs.search(q4_vector, num_results=5)

for i, r in enumerate(results4):
    print(f"{i+1}. {r['filename']}")

**Answer Q4:** The filename of the top result.

---
## Q5 — Text Search vs Vector Search

**Key concept:**  
- **Vector search** finds results by *meaning* — useful when the query and document use different words for the same idea.  
- **Text/keyword search** finds exact word matches — better when you need a specific term (like "pgvector" or a command name).  

We run both on the same query and compare which files appear in each.

In [ ]:
query5 = "How do I store vectors in PostgreSQL?"
q5_vector = embedder.encode(query5)

# Vector search
vector_results5 = vs.search(q5_vector, num_results=5)
vector_files5 = [r["filename"] for r in vector_results5]
print("Vector search top 5:")
for f in vector_files5:
    print(" ", f)

In [ ]:
# Text search — keyword matching
ts = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
ts.fit(chunks)

text_results5 = ts.search(query5, num_results=5)
text_files5 = [r["filename"] for r in text_results5]
print("Text search top 5:")
for f in text_files5:
    print(" ", f)

In [ ]:
# Files in vector results but NOT in text results
only_in_vector = set(vector_files5) - set(text_files5)
print("Only in vector results:", only_in_vector)

**Answer Q5:** The file that appears in vector results only.

---
## Q6 — Hybrid Search with RRF

**Key concept:** **Reciprocal Rank Fusion (RRF)** merges ranked lists from different search methods.  
Each document gets a score of `1 / (k + rank)` from every list it appears in — these scores are summed.  
- `k=60` is a standard default that prevents the very top result from dominating too much.  
- Documents that rank highly in *both* lists naturally bubble to the top.

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
query6 = "How do I give the model access to tools?"
q6_vector = embedder.encode(query6)

vector_results6 = vs.search(q6_vector, num_results=5)
text_results6 = ts.search(query6, num_results=5)

hybrid_results = rrf([vector_results6, text_results6])

print("Hybrid (RRF) top 5:")
for i, r in enumerate(hybrid_results):
    print(f"{i+1}. {r['filename']}")

**Answer Q6:** The top-ranked filename after fusion.